<a href="https://colab.research.google.com/github/BastianRu/Deep-Learning-Foundations-From-Scratch-Journal/blob/main/07-GPT_From_Scratch/GPT_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding="utf-8") as f:
    text = f.read()

print(f'Number of characters: {len(text):,}')

--2026-08-02 22:50:42--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-08-02 22:50:42 (22.0 MB/s) - ‘input.txt’ saved [1115394/1115394]

Number of characters: 1,115,394


In [43]:
#As in the previous architectures we're going to obtain the vocabulary and stoi/itos definitions
#But as we're working with a text instead of names, we cannot just define them the way we did before

# the set will give us the non-repeated characters in the text, then we create a list of those
# and lastly we use sorted.
chars = sorted(list(set(text)))
# define vocab size dinamically
vocab_size = len(chars)

# now we can define the stoi/itos using the same approach as before
stoi = { s:i for i, s in enumerate(chars) }
itos = { i:s for i,s in enumerate(chars) }

# but to build the dataset later, we're going to use so support functions
# A very simple implementation of a Tokenizer
encode = lambda s: [stoi[e] for e in s]
decode = lambda i: ''.join([itos[e] for e in i])

In [4]:
#Encoding the entire Shakespeare text
data = torch.tensor(encode(text), dtype=torch.long)
# We wrap the encoded list into a torch tensor
print(f"{(data.shape[0]):,}") #we get the same length

1,115,394


In [5]:
#Splitting the datasets into train, dev splits
nine_n = int(0.9*len(data))
train = data[:nine_n]
dev = data[nine_n:]

In [10]:
# The parallel processing

#generator just for reproducibility
torch.manual_seed = (1337)

# The same concept of batch size that we had before
batch_size = 4
# Previosuly, the block size defined the size of one single example
# But now, due to the new Transformer architecture, it also defines the number of sub-examples per each full sequence
# 7 sub-examples in a 8 block size sentence (excluding the 8th)
block_size = 8

def batch_split(split: str):
  data = train if split == "train" else dev #it's obvious what this does
  ix = torch.randint(len(data) - block_size, (batch_size,)) #We use a max value of the data length minus the blocksize
  x = torch.stack([ data[i:block_size+i] for i in ix ]) #torch.stack just stacks tensors along a given dimension (default 0) (it creates the batch dimension examples here)
  # for examples ix[0] is 79, the first example will be the characters from data tensor ranging from 79 to 79+8 = 87, so on so forth for all the random numbers in ix
  y = torch.stack([data[i+1:block_size+i+1] for i in ix]) #for the targets we just offset the indices
  return x, y

Xb, Yb = batch_split("train")

Xb, Yb


(tensor([[43, 52, 41, 53, 59, 52, 58, 43],
         [ 1, 57, 53, 52, 10,  0,  5, 32],
         [56, 51, 57, 12,  0, 15, 53, 51],
         [ 1, 58, 53,  1, 30, 53, 51, 43]]),
 tensor([[52, 41, 53, 59, 52, 58, 43, 56],
         [57, 53, 52, 10,  0,  5, 32, 61],
         [51, 57, 12,  0, 15, 53, 51, 43],
         [58, 53,  1, 30, 53, 51, 43,  0]]))

In [46]:
#Defining the simplest model to achieve this problem, the bigram model

class BigramLanguageModel(torch.nn.Module):
  def __init__(self, vocab_size):
    super().__init__() #I suppose this is for initialization of father class
    # Normally, we would pass the second parameter as the emb_dimensions to set up the number of features per word
    # But as we're building a bigram model, we need an exact matrix of shape (vocab_size, vocab_size) since we're looking for all the
    # possible combinations of characters. This is the lookup table C
    self.token_embedding_table = torch.nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    # idx and targets are both (B,T) tensor of integers
    # idx is equivalent to Xb and targets to Yb
    logits = self.token_embedding_table(idx) # (B,T,C) / This operation is equivalent to C[Xb]
    # after this with a block_size of 8 and a batch_size of 4, we perform (65, 65)[4, 8] = (4, 8, 65)

    if targets is None:
        loss = None
    else:
        B, T, C = logits.shape # We get the shape (4, 8, 65)
        logits = logits.view(B*T, C) # Here we basically stack up all the examples along the first dimension (deleting the second), we need this since the F.cross_entropy
        # requires the logits in a shape of (N, C) and the targets of (N,)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets) #We don't need to execute any linear pass as the model is just bigram

    return logits, loss

    # sampling from the model
  def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # The same "Xb" as we've seen before
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) #nn.Module allows us to define a forward pass and use it as it were  __call__()
            # focus only on the last time step since in INFERENCE we don't have targets, we're blind
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # append sampled index to the running sequence, we predicted the next character for each """example"""
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            # A curious fact is that we're supporting the batch dimension in inference, we usually want this
            # to generate several possible outcomes from a single input
        return idx

#Generation
# We're going to generate from a context of 1 word, and 1 character (zeroth character stands for a new line)
model = BigramLanguageModel(vocab_size)
context = torch.zeros((1, 1), dtype=torch.long)
encoded = model.generate(context, 100)
decoded = decode(encoded[0].tolist())
# We obviously we garbage since the model is not trained


'\nrXQE-vfflgD$;Jc xDeUWYJNbGrcTaxLUtFO:Ki;FFPg;O;XhLUwRlIIwZEteORfp$xBUp;;Ob.Rvr-US;:ZwRpPpQfpPTCpK&q;'